# Kaggle Proxy Score

## 1. Load Evaluator Bundle

In [ ]:
import io
import json
import os
import urllib.request
import numpy as np
import pandas as pd

BUNDLE_URL = 'https://raw.githubusercontent.com/AzrilFahmiardi/bank_csv/main/jointxinspire/kaggle_proxy_bundle.npz'
LOCAL_BUNDLE = '../output/kaggle_proxy/kaggle_proxy_bundle.npz'

def fetch_bundle():
    # Colab: download the latest bundle; local Jupyter: use the bundle exported by 04_kaggle_proxy_eval
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return LOCAL_BUNDLE
    urllib.request.urlretrieve(BUNDLE_URL, 'kaggle_proxy_bundle.npz')
    return 'kaggle_proxy_bundle.npz'

B = np.load(fetch_bundle())
META = json.loads(str(B['meta']))
IDS, S = B['ids'], B['scale'].astype(float)
NORMAL, MULT = B['normal'], B['mult'].astype(float)
SPECIAL, N = ~NORMAL, len(IDS)
print(f"bundle created {META['created']} | calibrated on {len(META['calibration_files'])} scored files"
      f" | LOO error mean {META['loo_mae']:.4f}, max {META['loo_max']:.4f}")

## 2. Scorer

In [ ]:
class PackedPT:
    # pseudo-truth stored as per-group quantile grids
    def __init__(self, prefix):
        self.lam = B[f'{prefix}_lam0'].astype(float) * MULT
        gid = B[f'{prefix}_gid']
        self.p = B[f'{prefix}_p']
        self.u = B[f'{prefix}_u'].astype(float)
        self.cs = np.concatenate([np.zeros((len(self.u), 1)), np.cumsum(self.u, axis=1)], axis=1)
        order = np.argsort(gid, kind='stable')
        bounds = np.searchsorted(gid[order], np.arange(len(self.p) + 1))
        self.groups = [order[bounds[g]:bounds[g + 1]] for g in range(len(self.p))]

    def expected_abs(self, a, zeta):
        out = np.empty(len(a))
        m = self.u.shape[1]
        for g, idx in enumerate(self.groups):
            if len(idx) == 0:
                continue
            pz = zeta * self.p[g] / (1 - self.p[g] + zeta * self.p[g])
            x = a[idx] / self.lam[idx]
            j = np.searchsorted(self.u[g], x, side='right')
            P = self.cs[g]
            e = (x * j - P[j] + (P[-1] - P[j]) - x * (m - j)) / m
            out[idx] = (1 - pz) * a[idx] + pz * self.lam[idx] * e
        return out

    def sample(self, zeta, rng):
        y = np.empty(N)
        for g, idx in enumerate(self.groups):
            if len(idx) == 0:
                continue
            pz = zeta * self.p[g] / (1 - self.p[g] + zeta * self.p[g])
            draw = self.u[g][rng.integers(0, self.u.shape[1], len(idx))]
            y[idx] = np.where(rng.random(len(idx)) < pz, draw, 0.0) * self.lam[idx]
        return y

ZETA = META['zeta']
PT_CONS = PackedPT('cons')
REF_SUB = B['ref_sub'].astype(float)
POOL = [(name, B[f'pool{i}_center'].astype(float), META['pool_wf'][i]) for i, name in enumerate(META['pool_names'])]
FAMILY = {}

def read_submission(src, name):
    s = pd.read_csv(src)
    assert {'id', 'total_ticket'} <= set(s.columns), f'{name}: needs columns id and total_ticket'
    s = s.sort_values('id').reset_index(drop=True)
    assert len(s) == N and (s.id.values == IDS).all(), f'{name}: ids must match test.csv ({N} rows)'
    v = s.total_ticket.to_numpy(dtype=float)
    assert np.isfinite(v).all() and (v >= 0).all(), f'{name}: values must be finite and non-negative'
    return v

def public_noise(a, b, n_rep=100):
    rng = np.random.default_rng(META['seed'])
    out = []
    for _ in range(n_rep):
        y = PT_CONS.sample(ZETA, rng)
        d = (np.abs(y - a) - np.abs(y - b)) / S
        mask = rng.random(N) < META['public_frac']
        out.append(d[mask].mean() - d.mean())
    return float(np.std(out))

def score(sub):
    dist = [float(np.mean(np.abs(sub - c)[NORMAL] / S[NORMAL])) for _, c, _ in POOL]
    k = int(np.argmin(dist))
    name, center, wf = POOL[k]
    if k not in FAMILY:
        pt = PackedPT(f'pool{k}')
        FAMILY[k] = (pt, pt.expected_abs(center, ZETA))
    pt, e_center = FAMILY[k]
    nd = float(((pt.expected_abs(sub, ZETA) - e_center) / S)[NORMAL].sum() / N)
    sp = float((PT_CONS.expected_abs(sub, ZETA) / S)[SPECIAL].sum() / N)
    est = META['c'] + META['beta'] * wf + nd + sp
    band = META['band_close'] if dist[k] < META['close_dist'] else META['band_new']
    vs = est - META['ref_fit']
    verdict = 'BETTER' if vs < -band else ('WORSE' if vs > band else 'TIE')
    return {'mase': est, 'band': band, 'vs_best': vs, 'verdict': verdict, 'nearest': name, 'dist': dist[k],
            'sd_public_vs_best': public_noise(sub, REF_SUB)}

def report(name, r):
    ref = META['ref']
    kind = ('normal-day predictions match a known model, only special days or post-processing changed'
            if r['dist'] < META['close_dist'] else
            f"normal-day predictions differ from every known model (nearest: {r['nearest']}, distance {r['dist']:.3f})")
    print('=' * 72)
    print(f'File                      : {name}')
    print(f"Estimated public LB MASE  : {r['mase']:.4f}")
    print(f"Likely range              : {r['mase'] - r['band']:.4f} to {r['mase'] + r['band']:.4f} (+/- {r['band']:.3f})")
    print(f"Versus best file {ref:.5f} : {r['vs_best']:+.4f} -> {r['verdict']}")
    print('-' * 72)
    print(f"- On {len(META['calibration_files'])} files with real Kaggle scores, this evaluator missed by "
          f"{META['loo_mae']:.3f} on average (worst {META['loo_max']:.3f}).")
    print(f"- This file: {kind}, so the range is +/- {r['band']:.3f}.")
    if r['dist'] > META['far_dist']:
        print('- Warning: very different from all known models, treat the estimate as rough.')
    print(f"- Separate from that: any single file's public and private LB can differ by about "
          f"+/- {META['public_level_sd']:.3f} (Kaggle sampling, not predictable).")
    print('- A difference inside the range is a tie: submit to know for sure.')

## 3. Upload CSV and Score

In [ ]:
try:
    from google.colab import files
    CSVS = {name: io.BytesIO(data) for name, data in files.upload().items() if name.lower().endswith('.csv')}
except ImportError:
    CSV_PATHS = []
    CSVS = {os.path.basename(p): p for p in CSV_PATHS}

results = []
for name, src in CSVS.items():
    r = score(read_submission(src, name))
    report(name, r)
    results.append({'file': name, **r})
if results:
    print('=' * 72)
    display(pd.DataFrame(results)[['file', 'mase', 'band', 'vs_best', 'verdict', 'nearest']].round(4))